# Entrenamiento YOLO11 para Belen Rumas en Colab

Este notebook entrena con `yolo11l.pt` oficial de Ultralytics, descarga Roboflow solo si el dataset no existe en Drive, y guarda `best.pt` en Google Drive.

Antes de ejecutar: `Runtime > Change runtime type > GPU`.

In [ ]:
!nvidia-smi
!python --version

In [ ]:
!pip install -q ultralytics==8.4.14 roboflow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import getpass
import os
import shutil

# Roboflow
ROBOFLOW_WORKSPACE = "asdcain-gzzzu"
ROBOFLOW_PROJECT = "belen"
ROBOFLOW_VERSION = 5
ROBOFLOW_FORMAT = "yolov8"  # compatible con entrenamiento YOLO11 en Ultralytics
FORCE_DOWNLOAD = False

# Entrenamiento
MODEL = "yolo11l.pt"       # modelo oficial; no usa model_detection.pt local
EPOCHS = 100
IMGSZ = 1024
BATCH = 2                  # T4: 1-2, L4: 2-4, A100: 4-8
WORKERS = 2
PATIENCE = 20
RUN_NAME = "rumas_yolo11l_colab"

# Rutas persistentes en Drive
DRIVE_ROOT = Path("/content/drive/MyDrive/belen_yolo")
DATASET_DIR = DRIVE_ROOT / "datasets" / f"roboflow_{ROBOFLOW_PROJECT}_v{ROBOFLOW_VERSION}"
RUNS_DIR = DRIVE_ROOT / "training_runs"
MODELS_DIR = DRIVE_ROOT / "models"

for path in (DRIVE_ROOT, DATASET_DIR.parent, RUNS_DIR, MODELS_DIR):
    path.mkdir(parents=True, exist_ok=True)

print("Dataset:", DATASET_DIR)
print("Runs:", RUNS_DIR)
print("Models:", MODELS_DIR)

In [ ]:
def get_roboflow_key():
    try:
        from google.colab import userdata
        key = userdata.get("ROBOFLOW_API_KEY")
    except Exception:
        key = None

    if not key:
        key = getpass.getpass("Roboflow API key: ")
    return key


def find_dataset_yaml(root: Path):
    if not root.exists():
        return None
    yamls = list(root.rglob("*.yaml")) + list(root.rglob("*.yml"))
    yamls = [p for p in yamls if "data" in p.name.lower() or "dataset" in p.name.lower()]
    if not yamls:
        return None
    yamls.sort(key=lambda p: (p.name.lower() != "data.yaml", len(p.relative_to(root).parts), str(p).lower()))
    return yamls[0]


def download_or_reuse_dataset():
    data_yaml = find_dataset_yaml(DATASET_DIR)
    if data_yaml and not FORCE_DOWNLOAD:
        print("Usando dataset local ya descargado:", data_yaml.parent)
        return data_yaml

    if DATASET_DIR.exists():
        print("Eliminando dataset previo/parcial:", DATASET_DIR)
        shutil.rmtree(DATASET_DIR)

    from roboflow import Roboflow
    rf = Roboflow(api_key=get_roboflow_key())
    project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
    version = project.version(ROBOFLOW_VERSION)

    print("Descargando Roboflow:", ROBOFLOW_PROJECT, "v", ROBOFLOW_VERSION, ROBOFLOW_FORMAT)
    try:
        dataset = version.download(ROBOFLOW_FORMAT, location=str(DATASET_DIR), overwrite=True)
    except TypeError:
        dataset = version.download(ROBOFLOW_FORMAT, location=str(DATASET_DIR))

    search_roots = [Path(getattr(dataset, "location", DATASET_DIR)), DATASET_DIR]
    for root in search_roots:
        data_yaml = find_dataset_yaml(root)
        if data_yaml:
            print("Dataset listo:", data_yaml)
            return data_yaml

    raise FileNotFoundError(f"No se encontro data.yaml dentro de {DATASET_DIR}")


DATA_YAML = download_or_reuse_dataset()
DATA_YAML

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)
results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    device=0,
    workers=WORKERS,
    cache=False,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    mosaic=1.0,
    mixup=0.1,
    close_mosaic=10,
    flipud=0.5,
    fliplr=0.5,
    degrees=15.0,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    perspective=0.0,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    lr0=0.01,
    lrf=0.001,
    warmup_epochs=3,
    cos_lr=True,
    optimizer="AdamW",
    weight_decay=0.0005,
    label_smoothing=0.1,
    amp=True,
    plots=True,
    save=True,
    save_period=10,
    val=True,
    verbose=True,
)

In [ ]:
from datetime import datetime

run_dir = Path(results.save_dir)
best_src = run_dir / "weights" / "best.pt"
if not best_src.exists():
    raise FileNotFoundError(best_src)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
dest_versioned = MODELS_DIR / f"model_detection_colab_{RUN_NAME}_{timestamp}.pt"
dest_latest = MODELS_DIR / "model_detection_colab.pt"
shutil.copy2(best_src, dest_versioned)
shutil.copy2(best_src, dest_latest)

print("Mejor modelo copiado a:", dest_versioned)
print("Ultimo modelo Colab:", dest_latest)
print("Run completo:", run_dir)

## Reanudar si Colab se corta

Ejecuta primero las celdas de instalacion, Drive y configuracion. Luego ejecuta esta celda en vez de la celda de entrenamiento normal.

In [ ]:
# Opcional: reanudar desde last.pt
from ultralytics import YOLO

last_ckpt = RUNS_DIR / RUN_NAME / "weights" / "last.pt"
if not last_ckpt.exists():
    raise FileNotFoundError(last_ckpt)

model = YOLO(str(last_ckpt))
results = model.train(resume=True)